# ViT Vanilla

## Imports e warnigns

In [ ]:
from BUSBRA_Medical import BUSBRADataModule
from classifiers import ClassificationModel
import pytorch_lightning as pl
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("lightning").addHandler(logging.NullHandler())
logging.getLogger("lightning").propagate = False

## Setando seeds para reprodutibilidade


In [ ]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    # Set the seed for Python's built-in random library
    random.seed(seed)
    
    # Set the seed for NumPy
    np.random.seed(seed)
    
    # Set the seed for PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    
    # For performance reasons, this setting should only be enabled for true reproducibility.
    # It can lead to slower training in some cases.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set the seed for PyTorch Lightning
    pl.seed_everything(seed, workers=True)

# Call the function to set seeds
set_seed(42)

In [ ]:
dm = BUSBRADataModule()

dm.setup('train')
dm.imshow_train()

## Carregando CSV do dataset

In [ ]:
import pandas as pd

merged_df = df = pd.read_csv('../busbra_medical_error.csv')
merged_df.head()

## Visualização do dataset

Importante para saber se o banco de dados está sendo carregado corretamente

In [ ]:
dm = BUSBRADataModule(Kfold=True, train=merged_df)
dm.imshow_train()

## Treino do modelo

In [ ]:
from sklearn.model_selection import KFold
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from matplotlib import pyplot as plt
import json
from datetime import datetime

moment = datetime.now().strftime("%d-%m-%y-%H-%M-%S")

Kfold = [1,2,3,4,5]

hp = [0.0001, 0.9, 0.0001, 10]

for i in Kfold:
    print(f'####### Fold {1} #######')
    column = 'valid_' + str(i)
    
    train_df = merged_df[merged_df[column] == 1]
    val_df = merged_df[merged_df[column] == 0]
    test_df = merged_df[merged_df['kFold'] == i]
    train_df.reset_index(drop=False,inplace=True)
    val_df.reset_index(drop=False,inplace=True)
    test_df.reset_index(drop=False,inplace=True)    

    count_classes = [len(train_df[train_df['Pathology'] == 'benign']), len(train_df[train_df['Pathology'] == 'malignant'])]
    class_weights = 1/torch.tensor(count_classes)
    class_weights = class_weights/torch.mean(class_weights)

    dm = BUSBRADataModule(Kfold=True, train=train_df, val=val_df, test=test_df)

    model_name = 'vit_b_16'
    model_hparams={"in_channels": 3, "num_classes": 2, "act_fn_name": "relu"}
    optimizer_name="SGD"
    optimizer_hparams={"lr": hp[0], "momentum": hp[1], "weight_decay": hp[2]}
    max_epochs = 150
    patience = 20
    
    early_stop_callback = EarlyStopping(
                            monitor="val_loss", 
                            patience=patience, 
                            verbose=False, 
                            mode="min"
                            )

    checkpoint_callback = ModelCheckpoint(
         dirpath='weights/'+model_name,
         filename=f'BUSBRA-{model_name}-fold-{i}-{moment}',
         monitor='val_loss',
         mode="min",
        )
    
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=1, 
        precision='16-mixed',
        max_epochs=max_epochs,
        callbacks = [early_stop_callback, checkpoint_callback],
        accumulate_grad_batches=2,
        )

    model = ClassificationModel(model_name, model_hparams, optimizer_name, 
                                optimizer_hparams, 
                                loss_weight=class_weights,
                               ) 
    
    trainer.fit(
        model=model, 
        datamodule=dm
        )

    # plot f1xloss de treino e validação
    plt.plot(model.history['train_f1_score'], label="train_f1_score")
    plt.plot(model.history['val_f1_score'], label="val_f1_score")
    plt.plot(model.history['train_loss'], label="train_loss")
    plt.plot(model.history['val_loss'], label="val_loss")
    plt.legend(bbox_to_anchor=(1.05, 1),
                             loc='upper left', borderaxespad=0.,
                             prop={'size': 20})
    plt.show()

In [ ]:
moment